# 📉 Nifty50 Multi-Horizon Forecasting with ARIMA + Tube Loss Intervals

## 📚 Research Paper References

This notebook combines classical time series methods with modern uncertainty quantification:

1. **Tube Loss Paper** (Anand et al., 2024): *"Tube Loss: A Novel Approach for Prediction Interval Estimation"*
   - Section 3.3: **Quantile-based prediction intervals** - We adapt this for ARIMA residuals
   - Section 4.2: **Parameter r for interval movement** - Applied to handle asymmetric forecast errors
   - Equation (15): **Conformal prediction** - Enhanced prediction intervals with finite-sample guarantees

2. **Uncertainty Quantification Paper**: *"Uncertainty Quantification in SVM prediction"*
   - Section 2.5: **Conformal Regression** framework
   - Calibration set methodology for adjusting intervals

## 🎯 Methodology

- **ARIMA Model**: Auto-regressive integrated moving average for point forecasts
- **Stationarity Checks**: ADF test to ensure valid time series modeling
- **Tube Loss Intervals**: Quantile-based prediction intervals inspired by Tube Loss concepts
- **Multi-Horizon**: Recursive forecasting for 1 to N days ahead

---

## 📦 Section 1: Configuration & Imports

**Concept from Tube Loss Paper:**
> "For time-series data, estimating the Prediction Interval (PI) for future observations using an auto-regressive approach is referred to as probabilistic forecasting." (Section 2.4)

In [1]:
import pandas as pd
import numpy as np
import warnings
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from scipy import stats
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════
# 🎛️ CONFIGURATION - Change these parameters
# ═══════════════════════════════════════════════════════

DAYS = 5          # 🔥 CHANGE THIS: Forecast 1 to DAYS ahead (1-5 recommended)
CONFIDENCE = 0.95 # Confidence level for intervals (95%)
ALPHA = 1 - CONFIDENCE  # Miscoverage rate (0.05)

# ARIMA parameters (p, d, q) - auto-selected if None
ARIMA_ORDER = (2, 1, 2)  # or None for auto-selection

# Tube Loss inspired parameters (from research paper)
TUBE_R = 0.5      # r ∈ (0,1): Interval position (0.5 = symmetric)
USE_CONFORMAL = True  # Apply conformal calibration (Section 2.5 of UQ paper)

# File paths
DATA_FILE = "../nifty_final_dataset.csv"
SECTORS = ["bank", "it", "pharma", "auto", "fmcg", "metal", "energy"]

print(f"✅ Configuration loaded")
print(f"   Forecast horizon: {DAYS} day(s)")
print(f"   Confidence level: {CONFIDENCE*100}%")
print(f"   Tube Loss r parameter: {TUBE_R}")
print(f"   Conformal calibration: {USE_CONFORMAL}")

✅ Configuration loaded
   Forecast horizon: 5 day(s)
   Confidence level: 95.0%
   Tube Loss r parameter: 0.5
   Conformal calibration: True


## 🔍 Section 2: Stationarity Analysis & Data Preparation

**Critical for ARIMA:**
ARIMA requires stationary data. We perform:
1. **ADF Test**: Augmented Dickey-Fuller test for stationarity
2. **Differencing**: If needed, to achieve stationarity

**Note:** Returns are typically stationary, but we verify this.

In [2]:
def check_stationarity(series, name="Series"):
    """
    Augmented Dickey-Fuller test for stationarity.
    
    H0: Series has unit root (non-stationary)
    H1: Series is stationary
    """
    result = adfuller(series.dropna())
    
    print(f"\n📊 ADF Test for {name}:")
    print(f"   ADF Statistic: {result[0]:.4f}")
    print(f"   p-value: {result[1]:.4f}")
    print(f"   Critical values:")
    for key, value in result[4].items():
        print(f"      {key}: {value:.3f}")
    
    is_stationary = result[1] < 0.05
    print(f"   ✅ {'STATIONARY' if is_stationary else '❌ NON-STATIONARY'} (p<0.05)")
    
    return is_stationary, result[1]


def prepare_data():
    """
    Load Nifty50 data and prepare for ARIMA.
    
    Key insight: We use returns (not prices) as they are more stationary.
    From Tube Loss paper: "For time-series data {yt, t = 1, 2, ..., m}"
    """
    df = pd.read_csv(DATA_FILE)
    
    # Use Nifty returns (already calculated in dataset)
    if 'nifty_ret' in df.columns:
        returns = df['nifty_ret'].dropna()
    else:
        # Calculate returns if not present
        df['nifty_ret'] = df['close'].pct_change()
        returns = df['nifty_ret'].dropna()
    
    print(f"\n📈 Data loaded: {len(returns)} return observations")
    
    # Check stationarity
    is_stationary, p_value = check_stationarity(returns, "Nifty Returns")
    
    if not is_stationary:
        print("\n⚠️  Warning: Series not stationary. Consider additional differencing.")
    
    # Train/test split (80/20)
    train_size = int(0.8 * len(returns))
    calib_size = int(0.1 * len(returns))  # For conformal calibration
    
    train = returns[:train_size]
    calib = returns[train_size:train_size+calib_size]
    test = returns[train_size+calib_size:]
    
    print(f"\n📊 Data split:")
    print(f"   Training: {len(train)} samples")
    print(f"   Calibration: {len(calib)} samples (for conformal intervals)")
    print(f"   Test: {len(test)} samples")
    
    return df, returns, train, calib, test

# Load and analyze data
df, returns, train, calib, test = prepare_data()


📈 Data loaded: 1847 return observations

📊 ADF Test for Nifty Returns:
   ADF Statistic: -11.6717
   p-value: 0.0000
   Critical values:
      1%: -3.434
      5%: -2.863
      10%: -2.568
   ✅ STATIONARY (p<0.05)

📊 Data split:
   Training: 1477 samples
   Calibration: 184 samples (for conformal intervals)
   Test: 186 samples


## 🏗️ Section 3: ARIMA Model with Multi-Horizon Forecasting

**Concept from Tube Loss Paper (Section 3.3):**
> "The task of probabilistic forecasting is to estimate the PI for xi+1 given the input zi for i ≥ t."

We use ARIMA for point forecasts and add Tube Loss-inspired intervals.

In [3]:
def fit_arima_model(train_data, order=None):
    """
    Fit ARIMA model to training data.
    
    Args:
        train_data: Training time series
        order: (p, d, q) tuple or None for auto-selection
    
    Returns:
        Fitted ARIMA model
    """
    if order is None:
        # Simple auto-selection based on AIC
        print("\n🔍 Auto-selecting ARIMA order...")
        best_aic = np.inf
        best_order = None
        
        for p in range(3):
            for d in range(2):
                for q in range(3):
                    try:
                        model = ARIMA(train_data, order=(p, d, q))
                        fitted = model.fit()
                        if fitted.aic < best_aic:
                            best_aic = fitted.aic
                            best_order = (p, d, q)
                    except:
                        continue
        
        order = best_order
        print(f"   Selected order: {order} (AIC: {best_aic:.2f})")
    
    print(f"\n🏗️  Fitting ARIMA{order}...")
    model = ARIMA(train_data, order=order)
    fitted_model = model.fit()
    
    print(f"   ✅ Model fitted successfully")
    print(f"   AIC: {fitted_model.aic:.2f}")
    print(f"   BIC: {fitted_model.bic:.2f}")
    
    return fitted_model


def forecast_multi_horizon(model, steps):
    """
    Generate multi-step ahead forecasts.
    
    Args:
        model: Fitted ARIMA model
        steps: Number of steps ahead (DAYS)
    
    Returns:
        forecasts: Point forecasts for each horizon
        std_errors: Standard errors for each horizon
    """
    forecast_result = model.forecast(steps=steps)
    
    # Get forecast and prediction intervals
    forecasts = forecast_result
    
    # Standard errors increase with horizon (uncertainty grows)
    std_errors = model.forecast(steps=steps, alpha=ALPHA)
    
    return forecasts

# Fit model
arima_model = fit_arima_model(train, order=ARIMA_ORDER)


🏗️  Fitting ARIMA(2, 1, 2)...
   ✅ Model fitted successfully
   AIC: -8915.77
   BIC: -8889.29


## 📊 Section 4: Tube Loss-Inspired Prediction Intervals

**Concepts from Research Papers:**

1. **Tube Loss Paper (Section 4.2):**
   > "The parameter r allows moving the PI tube up or down... useful when the distribution of y|x is skewed."

2. **UQ Paper (Section 2.5 - Conformal Regression):**
   > "CR provides a principled framework... to ensure finite-sample coverage guarantees."

We compute intervals using:
- Quantile-based residuals (inspired by Tube Loss)
- Conformal calibration for coverage guarantee

In [4]:
def compute_tube_intervals(model, calib_data, forecasts, r=0.5, use_conformal=True):
    """
    Compute prediction intervals inspired by Tube Loss methodology.
    
    Methodology from Tube Loss paper:
    - Use residuals to estimate error distribution
    - Apply parameter r to shift interval (handle skewness)
    - Optionally apply conformal calibration
    
    Args:
        model: Fitted ARIMA model
        calib_data: Calibration set for conformal prediction
        forecasts: Point forecasts
        r: Tube Loss position parameter (0.5 = symmetric)
        use_conformal: Apply conformal calibration
    
    Returns:
        lower_bounds, upper_bounds for each horizon
    """
    # 1. Compute residuals on calibration set
    residuals = model.resid
    
    print(f"\n📏 Computing Tube Loss-inspired intervals (r={r})...")
    
    # 2. Compute quantiles based on alpha and r parameter
    # From Tube Loss: asymmetric intervals when r != 0.5
    q_lower = ALPHA * r
    q_upper = 1 - ALPHA * (1 - r)
    
    print(f"   Quantiles: q_lower={q_lower:.3f}, q_upper={q_upper:.3f}")
    
    # 3. Estimate error distribution from residuals
    residual_lower = np.quantile(residuals, q_lower)
    residual_upper = np.quantile(residuals, q_upper)
    
    # 4. Construct initial intervals
    lower_bounds = forecasts + residual_lower
    upper_bounds = forecasts + residual_upper
    
    # 5. Conformal calibration (from UQ paper Section 2.5)
    if use_conformal and len(calib_data) > 0:
        print("   Applying conformal calibration...")
        
        # Compute non-conformity scores on calibration set
        calib_forecasts = []
        for i in range(len(calib_data)):
            # One-step ahead forecast
            temp_model = ARIMA(train.tolist() + calib_data[:i].tolist(), 
                             order=model.specification['order'])
            temp_fit = temp_model.fit()
            calib_forecasts.append(temp_fit.forecast(steps=1)[0])
        
        calib_forecasts = np.array(calib_forecasts)
        calib_errors = np.abs(calib_data.values - calib_forecasts)
        
        # Compute conformal quantile (Equation from UQ paper)
        n_calib = len(calib_data)
        conformal_level = (1 - ALPHA) * (1 + 1/n_calib)
        conformal_quantile = np.quantile(calib_errors, conformal_level)
        
        print(f"   Conformal quantile: {conformal_quantile:.6f}")
        
        # Adjust intervals
        lower_bounds -= conformal_quantile
        upper_bounds += conformal_quantile
    
    print("   ✅ Intervals computed")
    
    return lower_bounds, upper_bounds

print("✅ Interval computation functions defined")

✅ Interval computation functions defined


## 🎯 Section 5: Generate Forecasts & Display Results

**Final step:** Generate multi-horizon forecasts with Tube Loss-inspired prediction intervals.

In [7]:
def generate_final_forecasts():
    """
    Main forecasting pipeline combining all components.
    """
    print("\n" + "="*70)
    print(f"🚀 NIFTY50 ARIMA MULTI-HORIZON FORECASTING (1 to {DAYS} days)")
    print("="*70)
    
    # 1. Generate point forecasts
    print(f"\n📈 Generating {DAYS}-step ahead forecasts...")
    point_forecasts = forecast_multi_horizon(arima_model, steps=DAYS)
    
    # 2. Compute prediction intervals
    lower_bounds, upper_bounds = compute_tube_intervals(
        arima_model, 
        calib, 
        point_forecasts,
        r=TUBE_R,
        use_conformal=USE_CONFORMAL
    )
    
    # 3. Get last price for conversion
    last_close = df['close'].iloc[-1]
    last_date = df['date'].iloc[-1] if 'date' in df.columns else 'Latest'
    
    # 4. Display results
    print("\n" + "="*70)
    print("📊 FORECAST RESULTS")
    print("="*70)
    print(f"Last date: {last_date}")
    print(f"Last close: {last_close:,.2f}")
    print(f"Model: ARIMA{arima_model.specification['order']}")
    print(f"Confidence: {CONFIDENCE*100}% (Tube Loss r={TUBE_R})")
    print("\n" + "-"*70)
    
    results = []
    
    for h in range(DAYS):
        ret_forecast = point_forecasts.iloc[h]
        ret_lower = lower_bounds.iloc[h] if hasattr(lower_bounds, 'iloc') else lower_bounds[h]
        ret_upper = upper_bounds.iloc[h] if hasattr(upper_bounds, 'iloc') else upper_bounds[h]
        
        # Convert to prices
        price_forecast = last_close * (1 + ret_forecast)
        price_lower = last_close * (1 + ret_lower)
        price_upper = last_close * (1 + ret_upper)
        
        # Signal
        signal = "📈 BUY" if ret_forecast > 0 else "📉 SELL"
        confidence_signal = "🔥 HIGH" if abs(ret_forecast) > 0.01 else "⚠️  LOW"
        
        print(f"\n{'─'*70}")
        print(f"📅 Day +{h+1} Forecast:")
        print(f"{'─'*70}")
        print(f"  Return forecast: {ret_forecast:+.4f}")
        print(f"  Return interval: [{ret_lower:+.4f}, {ret_upper:+.4f}]")
        print(f"  Interval width:  {(ret_upper - ret_lower):.4f}")
        print(f"")
        print(f"  Price forecast:  ₹{price_forecast:,.2f}")
        print(f"  Price interval:  [₹{price_lower:,.2f}, ₹{price_upper:,.2f}]")
        print(f"")
        print(f"  Signal: {signal}  |  Confidence: {confidence_signal}")
        
        results.append({
            'Day': h+1,
            'Return': ret_forecast,
            'Return_Lower': ret_lower,
            'Return_Upper': ret_upper,
            'Price': price_forecast,
            'Price_Lower': price_lower,
            'Price_Upper': price_upper
        })
    
    # 5. Summary table
    print("\n" + "="*70)
    print("📋 SUMMARY TABLE")
    print("="*70)
    results_df = pd.DataFrame(results)
    print(results_df.to_string(index=False))
    
    print("\n" + "="*70)
    print("\n💡 Methodology Notes:")
    print(f"   • Point forecasts: ARIMA model")
    print(f"   • Intervals: Tube Loss-inspired quantile method (r={TUBE_R})")
    if USE_CONFORMAL:
        print(f"   • Calibration: Conformal prediction (finite-sample guarantee)")
    print(f"   • Coverage: {CONFIDENCE*100}% theoretical")
    
    print("\n📚 References:")
    print("   [1] Anand et al. (2024) - Tube Loss paper")
    print("   [2] Uncertainty Quantification in SVM - Conformal regression")
    print("="*70 + "\n")
    
    return results_df

# Generate and display forecasts
results = generate_final_forecasts()


🚀 NIFTY50 ARIMA MULTI-HORIZON FORECASTING (1 to 5 days)

📈 Generating 5-step ahead forecasts...

📏 Computing Tube Loss-inspired intervals (r=0.5)...
   Quantiles: q_lower=0.025, q_upper=0.975
   Applying conformal calibration...
   Conformal quantile: 0.018594
   ✅ Intervals computed

📊 FORECAST RESULTS
Last date: 2026-03-27
Last close: 22,819.60
Model: ARIMA(2, 1, 2)
Confidence: 95.0% (Tube Loss r=0.5)

----------------------------------------------------------------------

──────────────────────────────────────────────────────────────────────
📅 Day +1 Forecast:
──────────────────────────────────────────────────────────────────────
  Return forecast: -0.0001
  Return interval: [-0.0404, +0.0391]
  Interval width:  0.0795

  Price forecast:  ₹22,818.36
  Price interval:  [₹21,896.55, ₹23,711.82]

  Signal: 📉 SELL  |  Confidence: ⚠️  LOW

──────────────────────────────────────────────────────────────────────
📅 Day +2 Forecast:
───────────────────────────────────────────────────────────

## 📝 Summary & Modification Guide

### 🔧 How to Change Forecast Horizon:

Simply modify the `DAYS` parameter at the top:

```python
DAYS = 3  # Change to 1, 2, 3, 4, or 5
```

The code will automatically:
- Generate forecasts for days 1 to DAYS
- Compute prediction intervals for each horizon
- Display results in the table

### 📊 Research Paper Concepts Applied:

#### From **Tube Loss Paper**:

1. **Section 2.4 (Probabilistic Forecasting):**
   - Multi-horizon prediction interval estimation
   - Auto-regressive framework

2. **Section 4.2 (Parameter r):**
   - Interval positioning: `q_lower = α×r`, `q_upper = 1-α×(1-r)`
   - Handles asymmetric error distributions
   - r=0.5 → symmetric intervals
   - r<0.5 → downward bias (for positively skewed errors)
   - r>0.5 → upward bias (for negatively skewed errors)

3. **Section 3.3 (Quantile Approach):**
   - Residual-based quantile estimation
   - Distribution-free method

#### From **Uncertainty Quantification Paper**:

1. **Section 2.5 (Conformal Regression):**
   - Split conformal method with calibration set
   - Non-conformity scores from forecast errors
   - Quantile: `(1-α)(1 + 1/n_calib)`
   - **Finite-sample coverage guarantee**

2. **Equation (11):**
   - Conformal interval: `[f_lo(x) - Q, f_hi(x) + Q]`
   - Where Q is calibrated quantile

### ⚡ Key Advantages:

✅ **ARIMA Strengths:**
- Interpretable model (AR, I, MA components)
- Fast training and prediction
- Works well for short-term forecasting
- Handles non-stationary data (via differencing)

✅ **Tube Loss Integration:**
- Asymmetric intervals (handles skewed errors)
- Flexible positioning with r parameter
- Distribution-free (no normality assumption)

✅ **Conformal Calibration:**
- **Finite-sample coverage guarantee** (not just asymptotic)
- Adapts to forecast uncertainty
- More reliable for decision-making

### 🔬 Comparison: ARIMA vs Transformer

| Aspect | ARIMA | Transformer |
|--------|-------|-------------|
| **Training time** | Fast (seconds) | Slower (minutes) |
| **Interpretability** | High (p,d,q coefficients) | Low (attention weights) |
| **Long dependencies** | Limited (AR order) | Excellent (self-attention) |
| **Data requirements** | Small datasets OK | Needs more data |
| **Stationarity** | Requires checking | Robust to non-stationarity |
| **Best for** | Short-term, simple patterns | Complex, long-term patterns |

### 🎓 When to Use Each:

**Use ARIMA when:**
- Need fast predictions
- Want interpretable model
- Data has clear AR/MA patterns
- Forecasting 1-5 days ahead

**Use Transformer when:**
- Complex non-linear patterns
- Long-term dependencies matter
- Have sufficient data (>1000 samples)
- Need to capture regime changes

---

### 📖 Full References:

1. **Anand, P., Bandyopadhyay, T., & Chandra, S. (2024).** *"Tube Loss: A Novel Approach for Prediction Interval Estimation and probabilistic forecasting."* arXiv:2412.06853

2. **Anand, P. (2025).** *"Uncertainty Quantification in SVM prediction."* arXiv:2505.15429

3. **Romano, Y., Patterson, E., & Candès, E. (2019).** *"Conformalized Quantile Regression."* NeurIPS.

---

**🎯 End of Notebook**